<a href="https://colab.research.google.com/github/NadiaCarvalho/BroadcastJSB/blob/main/Create_Database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -O 'chords.npy' 'https://drive.usercontent.google.com/download?id=1bnChKI3ggvrDW0ffB2uHSLhy-aBuRVJZ&export=download&confirm=yes'
!wget -O 'latent.npy' 'https://drive.usercontent.google.com/download?id=10Wvs1KmwI-PGYbayXQq2sB-D3nZ0snGQ&export=download&confirm=yes'

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import json

# --- 1. Load Data from .npy files ---

# Load chord data (pitchclass and pianoroll)
chords_data = sorted(np.load('chords.npy', allow_pickle=True), key=lambda x: x[0])

# Load latent vectors (z)
# CRITICAL FIX: Ensure latent vectors are converted to lists immediately after loading
chords_in_lt = np.load('latent.npy', allow_pickle=True).tolist()

print(f'\nNumber of chords: {len(chords_data)}; In latent space: {len(chords_in_lt)}')

# --- 2. Combine Data into DataFrame ---

chords_all = pd.DataFrame(chords_data, columns=['pitchclass', 'pianoroll'])
chords_all.loc[:,'z'] = chords_in_lt

# --- 3. Add Required Fields for Vue Application ---

chords_all.loc[:,'id'] = chords_all.index.astype(str)

# --- Placeholder PCA Reduction Logic ---
# CRITICAL FIX: Ensure the output of this function is a standard Python list
def get_z2D_placeholder(z_vector):
    # If the vector is a NumPy array here, ensure the slice is converted to a list
    if len(z_vector) >= 2:
        return [float(z_vector[0]), float(z_vector[1])] # Use float() to ensure compatibility
    return [0.0, 0.0]

# If 'z' itself is still an ndarray column, you might need an extra conversion step:
# If the previous assignment didn't work, ensure 'z' elements are lists:
# chords_all.loc[:,'z'] = chords_all['z'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)

chords_all.loc[:,'z2D'] = chords_all['z'].apply(get_z2D_placeholder)

# --- 4. Export to JSON ---

# Convert the DataFrame to a list of dictionaries (records)
# We select the fields required by the Vue app logic
output_data = chords_all[['id', 'z', 'z2D', 'pitchclass']].to_dict('records')

# Final check: Iterate through the output_data and ensure all arrays are lists
# This is the safest way to guarantee JSON serialization
for record in output_data:
    if isinstance(record['z'], np.ndarray):
        record['z'] = record['z'].tolist()
    if isinstance(record['z2D'], np.ndarray):
        record['z2D'] = record['z2D'].tolist()
    # Also check pitchclass if they were originally arrays
    if isinstance(record['pitchclass'], np.ndarray):
        record['pitchclass'] = record['pitchclass'].tolist()

final_json = {
    "chords": output_data
}

# Write to file
json_file_path = 'chords_bach_all.json'
with open(json_file_path, 'w') as f:
    json.dump(final_json, f, indent=2)

print(f'\nSuccessfully exported {len(output_data)} chords to {json_file_path} in the required format.')

In [ ]:
import json
import re
from collections import defaultdict
import music21 as m21
from tqdm import tqdm

def load_chord_dictionary(dict_path):
    try:
        with open(dict_path, 'r') as f:
            data = json.load(f)
            return {c['pitchclass']: c['id'] for c in data['chords']}
    except Exception as e:
        print(f"Error loading dictionary: {e}")
        return {}

def clean_bwv_id(score_obj, index):
    try:
        if score_obj.metadata and score_obj.metadata.title:
            title = score_obj.metadata.title
            match = re.search(r'bwv(\d+)', title.lower())
            if match: return f"BWV_{match.group(1)}"
    except: pass
    return f"Chorale_{index}"

def get_chorale_info(score_obj, index):
    bwv_id = f"BWV_{index}" # Fallback
    full_name = f"Chorale {index}"

    if score_obj.metadata:
        # 1. Try to find the BWV number for the ID
        title_str = score_obj.metadata.title or ""
        match = re.search(r'bwv\s*(\d+)', title_str.lower())
        if match:
            bwv_id = f"BWV_{match.group(1)}"

        if score_obj.metadata.movementName:
          bwv_id = score_obj.metadata.movementName.replace(".mxl", "").replace("bwv", "BWV ")

        # 2. Try to get the actual Title
        # Some Bach chorales store the name in 'title', others in 'alternativeTitle'
        name = score_obj.metadata.title
        if not name or 'bwv' in name.lower():
            name = score_obj.metadata.alternativeTitle

        if name:
            full_name = name

    return bwv_id, full_name

def generate_phrases_slice_by_slice(dict_path, output_path, limit=None):
    pitch_to_id = load_chord_dictionary(dict_path)
    if not pitch_to_id:
        print("Aborting: Dictionary not found.")
        return

    all_phrases = []

    # Get the list of chorales
    chorale_list = list(m21.corpus.chorales.Iterator(1, 371, returnType='stream'))

    # Slice the list if a limit is provided for testing
    if limit:
        chorale_list = chorale_list[:limit]

    print(f"Starting extraction for {len(chorale_list)} chorales...")

    for i, chorale in enumerate(tqdm(chorale_list)):
        # addTies=False creates a new vertical slice at every rhythmic change
        chorale_c = chorale.chordify(addTies=False)

        sequence = []
        for c in chorale_c.recurse(classFilter='Chord'):
            # Match the exact MIDI pitch string logic
            pts = '-'.join([str(int(cp.ps)) for cp in c.pitches])

            chord_id = pitch_to_id.get(pts)
            if not chord_id:
                continue

            sequence.append({
                "id": chord_id,
                "duration": float(c.quarterLength),
                # Storing all indices so the audio engine can track specific voices
                "all_indices": list(range(len(c.pitches)))
            })

        if sequence:
            # Get both the technical ID and the friendly name
            bwv_id, real_name = get_chorale_info(chorale, i + 1)
            all_phrases.append({
                "id": bwv_id,
                "name": real_name,
                "sequence": sequence
            })

    with open(output_path, 'w') as f:
        json.dump(all_phrases, f, indent=2)

    print(f"\nSuccess! Exported {len(all_phrases)} phrases.")

# --- EXECUTION ---
# For a full run, set limit=None. For testing, use limit=10.
generate_phrases_slice_by_slice('chords_bach_all.json', 'chorales.json')